# Increase Resolution — distributed (tile-level) walkthrough

This notebook runs Increase Resolution **as a distributed pipeline**: the GPU-heavy per-tile inference is
split off so it can run across **your own fleet of worker machines**. It is fully self-contained — it sets
up CodeArtifact and installs the package here, same as `code_example.ipynb`.

The pipeline decomposes into three steps, and **each maps to an install extra / a role**:

| Step | Role | Runs on | Install |
|---|---|---|---|
| `split(image)` | **coordinator** | CPU box, no GPU | `increase-resolution[cpu]` |
| `TileWorker.infer(tile)` | **worker** | each GPU machine | `increase-resolution[gpu]` |
| `merge(tiles, layout)` | **coordinator** | CPU box, no GPU | `increase-resolution[cpu]` |

The tiles are independent, so where each is upscaled doesn't change the result — the distributed output is
**identical** to the single-call `IncreaseResolution.execute`.

**This notebook runs all three roles on one machine** (so it's runnable end-to-end), and therefore installs
`increase-resolution[all]`. In production you'd install only each machine's own extra.

**Required environment (not flexible):** NVIDIA **A10** (Ampere), **CUDA 11.7.1**, **TensorRT 8.4.1.5**,
Python **3.10** — for any machine that runs a worker (`[gpu]`/`[all]`). Prepare the runtime as in the
README's *Runtime setup* (NVIDIA base image `nvcr.io/nvidia/pytorch:22.07-py3` + `unset PYTHONPATH`/
`LD_LIBRARY_PATH`, the cuBLAS `LD_PRELOAD`). A pure coordinator (`[cpu]`) needs none of the GPU runtime.

**Prerequisite:** `BRIA_API_TOKEN` (for the CodeArtifact credential).

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`bria-increase-res`** repository. The same token works for every
role's install below.

In [ ]:
import os
import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")

resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-increase-res"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
result = resp.json()["result"]
CODE_ARTIFACT_PASSWORD = result["authorization_token"].strip()
print("CodeArtifact credential acquired. Expires:", result.get("expiration"))

## 2. Install — one extra per role

Each machine installs **only its role's extra** (the CodeArtifact token is the password; username `aws`):

```bash
# coordinator machine (runs split/merge — no GPU, no engine):
pip install --upgrade "increase-resolution[cpu]" --extra-index-url "$CA"

# worker machine (runs tile inference on the GPU — needs torch cu117 + nvidia-tensorrt):
pip install --upgrade "increase-resolution[gpu]" \
  --extra-index-url https://download.pytorch.org/whl/cu117 \
  --extra-index-url https://pypi.ngc.nvidia.com --extra-index-url "$CA"
```

This single-machine demo plays **all** roles, so it installs `[all]` (= `[cpu]` + `[gpu]` + the pipeline):

In [ ]:
import subprocess, sys
from urllib.parse import quote

CA = (
    "https://aws:" + quote(CODE_ARTIFACT_PASSWORD, safe="")
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-increase-res/simple/"
)
# [all] because this one box is coordinator AND worker. On a real coordinator use [cpu]; on a worker use [gpu].
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "increase-resolution[all]",
    "--extra-index-url", "https://download.pytorch.org/whl/cu117",
    "--extra-index-url", "https://pypi.ngc.nvidia.com",
    "--extra-index-url", CA,
])

## 3. Download the engine(s) — workers only

Only machines that run a **worker** (`[gpu]`/`[all]`) need the `.engine` files; a pure `[cpu]` coordinator
does not. Replace the URLs with the download links Bria gave you.

In [ ]:
from pathlib import Path

cache = Path(os.path.expanduser("~/.cache/bria/increase-resolution/"))
cache.mkdir(parents=True, exist_ok=True)

ENGINE_URLS = {
    2: "<BRIA_PROVIDED_URL_FOR_increase_resolution2.engine>",
    4: "<BRIA_PROVIDED_URL_FOR_increase_resolution4.engine>",
}
ENGINE_PATHS = {}
for scale, url in ENGINE_URLS.items():
    dest = cache / f"increase_resolution{scale}.engine"
    if not dest.exists():
        tmp = dest.with_suffix(".engine.part")
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
        tmp.rename(dest)
    ENGINE_PATHS[scale] = str(dest)
    print(f"x{scale} engine:", dest)

SCALE = 4
SAMPLE = "https://bria-test-images.s3.us-east-1.amazonaws.com/starry_2000x2000.png"

## 4. Split — coordinator (`[cpu]`)

`split` cuts the image into independent tiles (plain numpy arrays) plus a `layout` for reassembly and the
input alpha. **No GPU or engine here** — this is what the `[cpu]` coordinator runs. The tiles are what you
ship to your workers.

In [ ]:
from bria_external.image import ImageModel
from increase_resolution import split

image = ImageModel(image=SAMPLE).image
res = split(image, scale=SCALE)
print(f"split into {len(res.tiles)} tiles of shape {res.tiles[0].shape} -> send these to the [gpu] workers")

## 5. Tile inference — workers (`[gpu]`)

This is the **tile-only GPU processing** — all the `[gpu]` extra provides. Each worker machine builds a
`TileWorker` **once** and runs `infer` on the tiles it receives.

> **Replace the loop with your cross-machine transport.** In production you ship `res.tiles` to your GPU
> workers (queue / RPC / Ray / gRPC / …), each runs `TileWorker(engine).infer(tile)`, and you collect the
> upscaled tiles back **in order**. Here one local worker stands in for the fleet.

In [ ]:
from increase_resolution import TileWorker

# === your fan-out seam ===============================================================
worker = TileWorker(ENGINE_PATHS[SCALE]).setup()          # each worker machine, once
upscaled = [worker.infer(tile) for tile in res.tiles]     # <-- replace with your cross-machine dispatch
worker.close()
# =====================================================================================
print(f"upscaled {len(upscaled)} tiles on the worker")

## 6. Merge — coordinator (`[cpu]`)

Back on the coordinator, `merge` stitches the upscaled tiles into the final image (overlaps averaged). Again
**no GPU** — pure `[cpu]`.

In [ ]:
from IPython.display import display

from increase_resolution import merge

out = merge(upscaled, res.layout, res.alpha)
print("output size:", out.size)
out.save("output_distributed.png")
display(out)

## 7. Correctness — distributed == single-call

The distributed result must match `IncreaseResolution.execute` exactly — distributing the work doesn't change
the output. (This step needs the full pipeline, i.e. `[all]`.)

In [ ]:
import numpy as np

from increase_resolution import IncreaseResolution, IncreaseResolutionConfig, IncreaseResolutionInput

pipeline = IncreaseResolution(config=IncreaseResolutionConfig(engine_paths=ENGINE_PATHS))
pipeline.setup()
ref = pipeline.execute(IncreaseResolutionInput(image=image, desired_increase=SCALE)).image
pipeline.cleanup()

mae = np.abs(np.asarray(out).astype(int) - np.asarray(ref).astype(int)).mean()
print(f"distributed vs single-call MAE = {mae:.4f}/255  ({'MATCH' if mae < 1.0 else 'MISMATCH'})")